# Jaida Calvin — NLP Project Update 1
## Evaluating False Positives in AI Hate-Speech Detection

**Purpose:** address the professor's request for real fine-tuning, not just using released classifiers.

This notebook downloads the original datasets, saves a released-model baseline, updates the model's weights on HateXplain, and compares the resulting model on HateCheck. **It contains no precomputed research results.** Run one cell at a time, top to bottom.

**Before starting:** in Colab, choose **Runtime → Change runtime type → T4 GPU → Save** (or another available GPU). Use a fresh runtime. GPU availability is not guaranteed. You do not need Docker or a paid API key.

**Content warning:** the datasets contain hateful and offensive language. The notebook does not print raw posts by default. The two classes mean *hateful* and *not hateful under the dataset definition*; not-hateful does not necessarily mean appropriate or harmless.

**Important research rule:** keep the settings fixed during this first run. Do not change training settings in response to HateCheck results. Checkpoint selection uses HateXplain validation, not HateCheck.

### Step 1 — Install the text-model libraries
Click the triangle on the cell below. A running cell shows a stop square. Continue only when it finishes without a red traceback. These libraries are installed on Colab's computer, not your Mac. This pins Transformers to a known API version; it does not claim to be the latest release.

In [ ]:
import os, sys, subprocess
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "transformers==4.57.3", "librosa>=0.11,<0.12",
                       "scikit-learn>=1.4,<2", "requests>=2.31,<3", "tqdm>=4.66,<5"])
print("Step 1 complete: libraries installed.")

### Step 2 — Check the GPU and create the working folder
This is temporary Colab storage. Save the results ZIP before disconnecting. The saved notebook alone does not contain the large trained weights or all generated files.

In [ ]:
from pathlib import Path
import torch
PROJECT_DIR = Path("/content/jaida_nlp_update1")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "results").mkdir(exist_ok=True)
print("Working folder:", PROJECT_DIR)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT CONNECTED")
if not torch.cuda.is_available():
    print("Stop before training. Choose Runtime > Change runtime type > T4 GPU > Save.")
else:
    print("Step 2 complete: GPU is connected.")

### Step 3 — Load the project code
The full program is included in this notebook so there is no separate code file to upload. Run this cell without editing the long text. A readable copy is also included as `project.py` in the project ZIP.

The helper logic was tested locally on 12 synthetic unit tests. Those tests are NOT model experiments. The full dataset-download/training workflow still needs to run here.

In [ ]:
MODULE_SOURCE = '"""NLP Project Update 1: continued fine-tuning and false-positive evaluation.\n\nAuthor: Jaida Calvin (course project; implementation prepared with AI assistance).\nNo experimental outputs are bundled. Run in a fresh Google Colab GPU runtime.\nRaw dataset text and model weights are intentionally excluded from exports.\n"""\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any\nimport gc\nimport hashlib\nimport importlib.metadata\nimport io\nimport json\nimport math\nimport platform\nimport random\nimport re\nimport unicodedata\nimport zipfile\n\nimport numpy as np\nimport pandas as pd\nimport requests\nfrom requests.adapters import HTTPAdapter\nfrom urllib3.util.retry import Retry\nfrom sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score\n\nMODEL_IDS = {\n    "dehatebert": "Hate-speech-CNERG/dehatebert-mono-english",\n    "roberta": "cardiffnlp/twitter-roberta-base-hate-latest",\n}\nDEFAULTS = {\n    "seed": 42, "epochs": 2, "learning_rate": 2e-5,\n    "batch_size": 8, "eval_batch_size": 16, "gradient_accumulation": 2,\n    "max_length": 128, "weight_decay": 0.01, "warmup_ratio": 0.1,\n}\nLABEL_MAP = {"normal": 0, "offensive": 0, "hatespeech": 1}\n\n\ndef now_utc() -> str:\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef write_json(path: Path, data: Any) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(data, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")\n\n\ndef normal_key(text: str) -> str:\n    """For duplicate checks only, never a replacement for the model\'s input."""\n    return " ".join(unicodedata.normalize("NFKC", str(text)).casefold().split())\n\n\ndef majority_label(annotators: list[dict[str, Any]]) -> str | None:\n    labels = [str(a["label"]).strip().lower() for a in annotators]\n    if not labels or any(x not in LABEL_MAP for x in labels):\n        raise ValueError("Missing or unsupported original HateXplain label.")\n    label, count = Counter(labels).most_common(1)[0]\n    # Resolve the ORIGINAL three-way vote before applying the binary mapping.\n    return label if count > len(labels) / 2 else None\n\n\ndef binary_metrics(y_true: Any, scores: Any, threshold: float = 0.5) -> dict[str, Any]:\n    y = np.asarray(y_true, dtype=int)\n    p = np.asarray(scores, dtype=float)\n    if y.ndim != 1 or p.shape != y.shape or len(y) == 0:\n        raise ValueError("Labels and scores must be nonempty equal-length vectors.")\n    if not np.isin(y, [0, 1]).all() or not np.isfinite(p).all():\n        raise ValueError("Invalid labels or non-finite scores.")\n    if np.any(p < 0) or np.any(p > 1) or not 0 <= threshold <= 1:\n        raise ValueError("Scores and threshold must lie in [0, 1].")\n    pred = (p >= threshold).astype(int)\n    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel().tolist()\n    return {\n        "n": len(y), "threshold": float(threshold), "tn": tn, "fp": fp, "fn": fn, "tp": tp,\n        "accuracy": float(accuracy_score(y, pred)),\n        "precision_hate": float(precision_score(y, pred, zero_division=0)),\n        "recall_hate": float(recall_score(y, pred, zero_division=0)),\n        "macro_f1": float(f1_score(y, pred, labels=[0, 1], average="macro", zero_division=0)),\n        "fpr": fp / (fp + tn) if fp + tn else None,\n        "fnr": fn / (fn + tp) if fn + tp else None,\n    }\n\n\ndef http_session() -> requests.Session:\n    s = requests.Session()\n    retries = Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])\n    s.mount("https://", HTTPAdapter(max_retries=retries))\n    s.headers["User-Agent"] = "Jaida-Calvin-NLP-course-research"\n    return s\n\n\ndef get_bytes(url: str) -> bytes:\n    try:\n        r = http_session().get(url, timeout=90)\n        r.raise_for_status()\n        return r.content\n    except requests.RequestException as exc:\n        raise RuntimeError(f"Could not download {url}. Check the Colab connection and retry the cell.") from exc\n\n\ndef github_commit(repo: str, branch: str) -> str:\n    info = json.loads(get_bytes(f"https://api.github.com/repos/{repo}/commits/{branch}"))\n    sha = info.get("sha", "")\n    if not re.fullmatch(r"[0-9a-f]{40}", sha):\n        raise RuntimeError(f"Could not resolve a fixed revision for {repo}.")\n    return sha\n\n\ndef download_sources(root: Path) -> tuple[dict[str, Any], dict[str, Any], pd.DataFrame]:\n    raw_dir = root / "data" / "raw"\n    raw_dir.mkdir(parents=True, exist_ok=True)\n    lock_file = root / "results" / "resources_lock.json"\n    lock = json.loads(lock_file.read_text()) if lock_file.exists() else {"created_utc": now_utc(), "sources": {}}\n    specs = {\n        "hatexplain": ("hate-alert/HateXplain", "master", "Data/dataset.json", "hatexplain.json"),\n        "hatexplain_splits": ("hate-alert/HateXplain", "master", "Data/post_id_divisions.json", "splits.json"),\n        "hatecheck": ("paul-rottger/hatecheck-data", "main", "test_suite_cases.csv", "hatecheck.csv"),\n    }\n    revisions: dict[str, str] = {}\n    for name, (repo, branch, rel, filename) in specs.items():\n        saved = lock["sources"].get(name)\n        if saved:\n            sha = saved["revision"]\n            revisions[repo] = sha\n        else:\n            if repo not in revisions:\n                revisions[repo] = github_commit(repo, branch)\n            sha = revisions[repo]\n        url = f"https://raw.githubusercontent.com/{repo}/{sha}/{rel}"\n        dest = raw_dir / filename\n        payload = dest.read_bytes() if dest.exists() else get_bytes(url)\n        digest = hashlib.sha256(payload).hexdigest()\n        if saved and saved["sha256"] != digest:\n            raise RuntimeError(f"Checksum changed for cached {name}; do not mix dataset versions.")\n        dest.write_bytes(payload)\n        lock["sources"][name] = {"url": url, "revision": sha, "sha256": digest}\n        write_json(lock_file, lock)\n    return (\n        json.loads((raw_dir / "hatexplain.json").read_text(encoding="utf-8")),\n        json.loads((raw_dir / "splits.json").read_text(encoding="utf-8")),\n        pd.read_csv(raw_dir / "hatecheck.csv"),\n    )\n\n\ndef parse_hatexplain(raw: dict[str, Any], divisions: dict[str, list[str]]) -> tuple[dict[str, pd.DataFrame], dict[str, Any]]:\n    frames, audit = {}, {}\n    seen_ids: set[str] = set()\n    for split in ("train", "val", "test"):\n        ids = divisions[split]\n        if len(set(ids)) != len(ids) or seen_ids.intersection(ids):\n            raise ValueError("Official split IDs are duplicated or overlap.")\n        seen_ids.update(ids)\n        rows, no_majority, empty = [], 0, 0\n        for post_id in ids:\n            item = raw[post_id]\n            label = majority_label(item["annotators"])\n            if label is None:\n                no_majority += 1\n                continue\n            text = " ".join(item["post_tokens"])\n            if not text.strip():\n                empty += 1\n                continue\n            rows.append({"id": str(post_id), "text": text, "label_original": label,\n                         "label": LABEL_MAP[label], "key": normal_key(text)})\n        frame = pd.DataFrame(rows)\n        if frame.empty:\n            raise ValueError(f"No usable examples in {split}.")\n        # Drop all conflicting duplicates; keep only one of identical same-label texts.\n        conflicts = frame.groupby("key")["label_original"].nunique()\n        conflict_keys = set(conflicts[conflicts > 1].index)\n        cleaned = frame[~frame["key"].isin(conflict_keys)].copy()\n        n_conflicts = len(frame) - len(cleaned)\n        n_dupes = int(cleaned.duplicated("key").sum())\n        cleaned = cleaned.drop_duplicates("key", keep="first").reset_index(drop=True)\n        frames[split] = cleaned\n        audit[split] = {"official_ids": len(ids), "excluded_no_majority": no_majority,\n                        "excluded_empty": empty, "excluded_conflicting_rows": n_conflicts,\n                        "removed_within_split_duplicates": n_dupes}\n    return frames, audit\n\n\ndef prepare_hatecheck(raw: pd.DataFrame) -> pd.DataFrame:\n    required = {"case_id", "test_case", "label_gold", "functionality", "target_ident", "templ_id"}\n    if not required.issubset(raw.columns):\n        raise ValueError(f"HateCheck is missing required columns: {required - set(raw.columns)}")\n    df = raw.copy()\n    mapping = {"non-hateful": 0, "hateful": 1}\n    df["label"] = df["label_gold"].map(mapping)\n    if df["label"].isna().any() or df["case_id"].duplicated().any() or df["test_case"].isna().any():\n        raise ValueError("HateCheck contains unknown labels, duplicate IDs, or missing text.")\n    if len(df) != 3728 or df["functionality"].nunique() != 29:\n        raise ValueError("Unexpected HateCheck version/size; review before proceeding.")\n    df["label"] = df["label"].astype(int)\n    df["id"] = df["case_id"].astype(str)\n    df["text"] = df["test_case"].astype(str)\n    df["key"] = df["text"].map(normal_key)\n    return df.reset_index(drop=True)\n\n\ndef isolate_splits(frames: dict[str, pd.DataFrame], hc: pd.DataFrame) -> tuple[dict[str, pd.DataFrame], dict[str, int]]:\n    """Preserve HateCheck, then test, then validation, removing overlaps from lower-priority sets."""\n    out, removed = {}, {}\n    forbidden = set(hc["key"])\n    for split in ("test", "val", "train"):\n        df = frames[split]\n        kept = df[~df["key"].isin(forbidden)].copy().reset_index(drop=True)\n        removed[split] = len(df) - len(kept)\n        if len(kept) == 0 or set(kept["label"]) != {0, 1}:\n            raise ValueError(f"Both binary classes are required after filtering in {split}.")\n        out[split] = kept\n        forbidden.update(kept["key"])\n    return out, removed\n\n\ndef prepare_data(root: str | Path) -> tuple[dict[str, pd.DataFrame], pd.DataFrame, dict[str, Any]]:\n    root = Path(root)\n    data, divisions, raw_hc = download_sources(root)\n    hc = prepare_hatecheck(raw_hc)\n    frames, audit = parse_hatexplain(data, divisions)\n    frames, removed = isolate_splits(frames, hc)\n    summary = []\n    for split in ("train", "val", "test"):\n        df = frames[split]\n        audit[split]["removed_cross_split_or_hatecheck_overlap"] = removed[split]\n        audit[split]["final_n"] = len(df)\n        summary.append({"dataset": "HateXplain", "split": split, "n": len(df),\n                        "non_hate": int((df["label"] == 0).sum()), "hate": int((df["label"] == 1).sum())})\n        # Save IDs and labels, not the raw post text, in the shareable results folder.\n        dest = root / "results" / f"hatexplain_{split}_ids.csv"\n        dest.parent.mkdir(parents=True, exist_ok=True)\n        df[["id", "label_original", "label"]].to_csv(dest, index=False)\n    summary.append({"dataset": "HateCheck", "split": "evaluation", "n": len(hc),\n                    "non_hate": int((hc["label"] == 0).sum()), "hate": int((hc["label"] == 1).sum())})\n    audit["hatecheck"] = {"n": len(hc), "functionalities": int(hc["functionality"].nunique()),\n                           "used_for_training": False, "used_for_checkpoint_selection": False}\n    write_json(root / "results" / "data_audit.json", audit)\n    pd.DataFrame(summary).to_csv(root / "results" / "dataset_counts.csv", index=False)\n    print(pd.DataFrame(summary).to_string(index=False))\n    print("Data preparation complete. These counts are not model performance results.")\n    return frames, hc, audit\n\n\ndef positive_class_index(id2label: dict[Any, str]) -> int:\n    normalized = {int(i): re.sub(r"[^a-z]", "", str(v).lower()) for i, v in id2label.items()}\n    positives = [i for i, text in normalized.items() if text in {"hate", "hateful", "hatespeech"}]\n    negatives = [i for i, text in normalized.items() if text in {"nonhate", "nothate", "nonhateful", "nothateful", "nonhatespeech"}]\n    if len(normalized) != 2 or len(positives) != 1 or len(negatives) != 1:\n        raise ValueError(f"Unverified class mapping {id2label}. Stop rather than guessing.")\n    return positives[0]\n\n\ndef seed_everything(seed: int) -> None:\n    import torch\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n    torch.use_deterministic_algorithms(True, warn_only=True)\n\n\ndef environment() -> dict[str, Any]:\n    import torch\n    libs = {}\n    for name in ("torch", "transformers", "huggingface_hub", "numpy", "pandas", "scikit-learn", "requests"):\n        try:\n            libs[name] = importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError:\n            libs[name] = "not installed"\n    return {"python": platform.python_version(), "libraries": libs,\n            "cuda": torch.cuda.is_available(),\n            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}\n\n\ndef model_revision(root: Path, model_id: str) -> str:\n    from huggingface_hub import HfApi\n    path = root / "results" / "model_revisions.json"\n    saved = json.loads(path.read_text()) if path.exists() else {}\n    if model_id not in saved:\n        saved[model_id] = HfApi().model_info(model_id).sha\n        write_json(path, saved)\n    return saved[model_id]\n\n\ndef token_dataset(frame: pd.DataFrame, tokenizer: Any, max_length: int, hate_index: int):\n    import torch\n    enc = tokenizer(frame["text"].tolist(), truncation=True, max_length=max_length, padding=False)\n    labels = [hate_index if int(x) == 1 else 1 - hate_index for x in frame["label"]]\n    class EncodedDataset(torch.utils.data.Dataset):\n        def __len__(self):\n            return len(labels)\n        def __getitem__(self, idx):\n            return {**{k: v[idx] for k, v in enc.items()}, "labels": labels[idx]}\n    return EncodedDataset()\n\n\ndef make_loader(dataset: Any, tokenizer: Any, batch_size: int, shuffle: bool, seed: int):\n    import torch\n    from transformers import DataCollatorWithPadding\n    return torch.utils.data.DataLoader(\n        dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0,\n        collate_fn=DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8),\n        generator=torch.Generator().manual_seed(seed),\n    )\n\n\ndef infer_scores(model: Any, loader: Any, device: Any, hate_index: int) -> np.ndarray:\n    import torch\n    from tqdm.auto import tqdm\n    model.eval()\n    values = []\n    with torch.inference_mode():\n        for batch in tqdm(loader, desc="Predicting", leave=False):\n            inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}\n            logits = model(**inputs).logits\n            values.extend(torch.softmax(logits.float(), dim=-1)[:, hate_index].cpu().tolist())\n    return np.asarray(values)\n\n\ndef save_evaluation(root: Path, model_key: str, condition: str, hc: pd.DataFrame, scores: np.ndarray) -> dict[str, Any]:\n    prefix = f"{model_key}_{condition}"\n    out = root / "results"\n    out.mkdir(parents=True, exist_ok=True)\n    columns = ["id", "label", "functionality", "target_ident", "templ_id"]\n    pred = hc[columns].copy()\n    pred["hate_score"] = scores\n    pred["prediction"] = (scores >= 0.5).astype(int)\n    pred.to_csv(out / f"{prefix}_predictions.csv", index=False)\n    m = binary_metrics(hc["label"], scores)\n    write_json(out / f"{prefix}_metrics.json", m)\n    rows = []\n    for functionality, indexes in hc.groupby("functionality").groups.items():\n        ix = np.asarray(list(indexes), dtype=int)\n        cm = binary_metrics(hc.loc[ix, "label"], scores[ix])\n        rows.append({"functionality": functionality, **{k: cm[k] for k in ("n", "accuracy", "tn", "fp", "fn", "tp", "fpr", "fnr")}})\n    pd.DataFrame(rows).to_csv(out / f"{prefix}_categories.csv", index=False)\n    pd.DataFrame([binary_metrics(hc["label"], scores, t) for t in np.linspace(0.1, 0.9, 9)]).to_csv(\n        out / f"{prefix}_thresholds.csv", index=False)\n    # Error IDs only in shareable outputs. Join locally with HateCheck to review text.\n    pred.loc[(pred.label == 0) & (pred.prediction == 1)].to_csv(out / f"{prefix}_false_positive_ids.csv", index=False)\n    return m\n\n\ndef run_model(root: str | Path, frames: dict[str, pd.DataFrame], hc: pd.DataFrame,\n              model_key: str = "dehatebert", config: dict[str, Any] | None = None) -> dict[str, Any]:\n    """Evaluate released checkpoint, update ALL parameters, select on validation, evaluate again.\n\n    HateCheck metrics must not be used to change this run\'s hyperparameters.\n    Any repeated run is a new experiment and must be documented, not silently overwritten.\n    """\n    import torch\n    from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup\n    from tqdm.auto import tqdm\n    root = Path(root)\n    cfg = {**DEFAULTS, **(config or {})}\n    if model_key not in MODEL_IDS:\n        raise ValueError(f"Choose one of {list(MODEL_IDS)}.")\n    if not torch.cuda.is_available():\n        raise RuntimeError("A GPU was not detected. In Colab choose Runtime > Change runtime type > T4 GPU, save, then rerun setup.")\n    if tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2]) < (2, 6):\n        raise RuntimeError("Use a current Colab runtime with PyTorch 2.6+; do not disable model-loading safety checks.")\n    output = root / "results"\n    output.mkdir(parents=True, exist_ok=True)\n    run_dir = output / model_key\n    if (run_dir / "status.json").exists():\n        existing = json.loads((run_dir / "status.json").read_text())\n        if existing.get("status") == "completed":\n            print(f"{model_key} already completed. Returning saved metrics without retraining.")\n            return json.loads((output / f"{model_key}_finetuned_metrics.json").read_text())\n        raise RuntimeError("A previous attempt exists. Export its outputs before starting a separate documented run.")\n    run_dir.mkdir(parents=True, exist_ok=True)\n    seed_everything(cfg["seed"])\n    model_id = MODEL_IDS[model_key]\n    revision = model_revision(root, model_id)\n    metadata = {"started_utc": now_utc(), "model_id": model_id, "revision": revision,\n                "config": cfg, "environment": environment(),\n                "train_n": len(frames["train"]), "validation_n": len(frames["val"]),\n                "hatecheck_n": len(hc), "condition": "continued full-parameter fine-tuning",\n                "selection": "highest HateXplain validation macro-F1 among the two trained epochs; earlier epoch wins ties",\n                "caveat": "Released checkpoints may have seen HateXplain previously; validation is not claimed independent of upstream training."}\n    write_json(run_dir / "run_config.json", metadata)\n    write_json(run_dir / "status.json", {"status": "started", "utc": now_utc(), "epochs_completed": 0})\n    model = tokenizer = optimizer = scheduler = None\n    try:\n        tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision, trust_remote_code=False)\n        model = AutoModelForSequenceClassification.from_pretrained(model_id, revision=revision, trust_remote_code=False)\n        if model.config.pad_token_id is None:\n            model.config.pad_token_id = tokenizer.pad_token_id\n        hate_index = positive_class_index(model.config.id2label)\n        metadata["id2label"] = {str(k): v for k, v in model.config.id2label.items()}\n        metadata["hate_index"] = hate_index\n        metadata["truncated_examples"] = {}\n        for name, df in {"train": frames["train"], "val": frames["val"], "hatecheck": hc}.items():\n            lengths = tokenizer(df.text.tolist(), truncation=False, padding=False, return_length=True)["length"]\n            metadata["truncated_examples"][name] = int(np.sum(np.asarray(lengths) > cfg["max_length"]))\n        datasets = {name: token_dataset(df, tokenizer, cfg["max_length"], hate_index)\n                    for name, df in {"train": frames["train"], "val": frames["val"], "hatecheck": hc}.items()}\n        train_loader = make_loader(datasets["train"], tokenizer, cfg["batch_size"], True, cfg["seed"])\n        val_loader = make_loader(datasets["val"], tokenizer, cfg["eval_batch_size"], False, cfg["seed"])\n        hc_loader = make_loader(datasets["hatecheck"], tokenizer, cfg["eval_batch_size"], False, cfg["seed"])\n        device = torch.device("cuda")\n        model.to(device)\n        baseline_scores = infer_scores(model, hc_loader, device, hate_index)\n        save_evaluation(root, model_key, "released", hc, baseline_scores)\n        print("Released-model outputs saved. Do not change hyperparameters based on these test results.")\n        for p in model.parameters():\n            p.requires_grad_(True)\n        metadata["trainable_parameters"] = sum(p.numel() for p in model.parameters() if p.requires_grad)\n        write_json(run_dir / "run_config.json", metadata)\n        # Full network updates, not just fitting the final classification head.\n        model.gradient_checkpointing_enable()\n        no_decay = ("bias", "LayerNorm.weight", "layer_norm.weight")\n        groups = [\n            {"params": [p for n, p in model.named_parameters() if not any(s in n for s in no_decay)], "weight_decay": cfg["weight_decay"]},\n            {"params": [p for n, p in model.named_parameters() if any(s in n for s in no_decay)], "weight_decay": 0.0},\n        ]\n        optimizer = torch.optim.AdamW(groups, lr=cfg["learning_rate"])\n        steps_per_epoch = math.ceil(len(train_loader) / cfg["gradient_accumulation"])\n        total_steps = steps_per_epoch * cfg["epochs"]\n        scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * cfg["warmup_ratio"]), total_steps)\n        scaler = torch.amp.GradScaler("cuda")\n        best_f1, history = -1.0, []\n        best_path = root / "checkpoints" / model_key / "best"\n        optimizer.zero_grad(set_to_none=True)\n        for epoch in range(1, cfg["epochs"] + 1):\n            model.train()\n            total_loss, example_count = 0.0, 0\n            bar = tqdm(train_loader, desc=f"{model_key}: training epoch {epoch}/{cfg[\'epochs\']}")\n            for step, batch in enumerate(bar):\n                batch = {k: v.to(device) for k, v in batch.items()}\n                # Correct normalization for the last, potentially incomplete accumulation group.\n                start = (step // cfg["gradient_accumulation"]) * cfg["gradient_accumulation"]\n                accumulation_group = min(cfg["gradient_accumulation"], len(train_loader) - start)\n                with torch.autocast(device_type="cuda", dtype=torch.float16):\n                    raw_loss = model(**batch).loss\n                    loss = raw_loss / accumulation_group\n                if not torch.isfinite(loss):\n                    raise RuntimeError("Non-finite training loss; do not report this attempt as successful.")\n                scaler.scale(loss).backward()\n                n = len(batch["labels"])\n                total_loss += float(raw_loss.detach()) * n\n                example_count += n\n                if (step + 1) % cfg["gradient_accumulation"] == 0 or step + 1 == len(train_loader):\n                    scaler.unscale_(optimizer)\n                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n                    old_scale = scaler.get_scale()\n                    scaler.step(optimizer)\n                    scaler.update()\n                    if scaler.get_scale() >= old_scale:\n                        scheduler.step()\n                    optimizer.zero_grad(set_to_none=True)\n                bar.set_postfix(loss=f"{total_loss / example_count:.4f}")\n            val_scores = infer_scores(model, val_loader, device, hate_index)\n            val_metrics = binary_metrics(frames["val"]["label"], val_scores)\n            row = {"epoch": epoch, "train_loss": total_loss / example_count,\n                   "validation_macro_f1": val_metrics["macro_f1"], "validation_accuracy": val_metrics["accuracy"]}\n            history.append(row)\n            pd.DataFrame(history).to_csv(run_dir / "training_history.csv", index=False)\n            if val_metrics["macro_f1"] > best_f1:\n                best_f1 = val_metrics["macro_f1"]\n                metadata["selected_epoch"] = epoch\n                model.save_pretrained(best_path, safe_serialization=True)\n                tokenizer.save_pretrained(best_path)\n                write_json(run_dir / "selected_validation_metrics.json", val_metrics)\n                write_json(run_dir / "run_config.json", metadata)\n            write_json(run_dir / "status.json", {"status": "training", "epochs_completed": epoch, "utc": now_utc()})\n            print(f"Epoch {epoch}: validation macro-F1 = {val_metrics[\'macro_f1\']:.4f}")\n        del optimizer, scheduler\n        optimizer = scheduler = None\n        model.cpu()\n        del model\n        model = None\n        gc.collect()\n        torch.cuda.empty_cache()\n        model = AutoModelForSequenceClassification.from_pretrained(best_path, trust_remote_code=False).to(device)\n        ft_scores = infer_scores(model, hc_loader, device, hate_index)\n        ft = save_evaluation(root, model_key, "finetuned", hc, ft_scores)\n        write_json(run_dir / "status.json", {"status": "completed", "epochs_completed": cfg["epochs"],\n                                             "selected_epoch": metadata["selected_epoch"], "completed_utc": now_utc()})\n        write_summary(root)\n        print(f"Completed {model_key}. Real measurements are in results/. No improvement is assumed.")\n        return ft\n    except Exception as exc:\n        prev = json.loads((run_dir / "status.json").read_text())\n        write_json(run_dir / "status.json", {**prev, "status": "interrupted_or_failed", "error": str(exc), "utc": now_utc()})\n        raise\n    finally:\n        if model is not None:\n            model.cpu()\n        del model, optimizer, scheduler\n        gc.collect()\n        torch.cuda.empty_cache()\n\n\ndef write_summary(root: str | Path) -> pd.DataFrame:\n    root = Path(root)\n    rows = []\n    for key in MODEL_IDS:\n        for condition in ("released", "finetuned"):\n            path = root / "results" / f"{key}_{condition}_metrics.json"\n            if path.exists():\n                rows.append({"model": key, "condition": condition, **json.loads(path.read_text())})\n    table = pd.DataFrame(rows)\n    if table.empty:\n        print("No model measurements yet. A missing result is not a score of zero.")\n        return table\n    table.to_csv(root / "results" / "model_comparison.csv", index=False)\n    cols = ["model", "condition", "n", "accuracy", "macro_f1", "fpr", "fnr"]\n    lines = ["# Measured results for Project Update 1", "", "Only completed evaluation outputs are listed. Scores are fractions, not percentages.", "",\n             "```", table[cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"), "```", "",\n             "The labels are HateCheck\'s annotations. These are benchmark results, not platform-level error rates.",\n             "HateCheck was not used for this project\'s training/checkpoint selection. Threshold 0.5 was fixed in advance.", ""]\n    for key in MODEL_IDS:\n        part = table[table.model == key].set_index("condition")\n        if {"released", "finetuned"}.issubset(part.index):\n            delta_fp = 100 * (part.loc["finetuned", "fpr"] - part.loc["released", "fpr"])\n            delta_fn = 100 * (part.loc["finetuned", "fnr"] - part.loc["released", "fnr"])\n            lines.append(f"{key}: fine-tuned minus released FPR = {delta_fp:+.2f} percentage points; FNR = {delta_fn:+.2f} percentage points. Interpret BOTH changes; neither improvement is assumed.")\n        else:\n            lines.append(f"{key}: paired before/after comparison is not complete.")\n    lines += ["", "Still pending unless separately documented: matched-identity bootstrap analysis, manual error review, repeated training seeds, and publication of the GitHub repository.",\n              "Cardiff\'s original training already included HateXplain. Do not call its HateXplain validation/test scores independent unseen-data results."]\n    (root / "results" / "results_for_report.md").write_text("\\n".join(lines), encoding="utf-8")\n    print(table[cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))\n    return table\n\n\ndef export_results(root: str | Path) -> Path:\n    root = Path(root)\n    write_summary(root)\n    result_dir = root / "results"\n    if not result_dir.exists():\n        raise RuntimeError("No results folder exists yet; run data preparation first.")\n    stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")\n    archive = root.parent / f"Jaida_Update1_Results_{stamp}.zip"\n    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:\n        for file in result_dir.rglob("*"):\n            if file.is_file():\n                z.write(file, file.relative_to(root))\n        if (root / "project.py").exists():\n            z.write(root / "project.py", "project.py")\n    print(f"Saved {archive.name}. Raw texts and large model weights are not included.")\n    return archive\n'
(PROJECT_DIR / "project.py").write_text(MODULE_SOURCE, encoding="utf-8")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
import importlib
import project
importlib.reload(project)
print("Step 3 complete: project code loaded.")

### Step 4 — Download and check the datasets
HateXplain provides training and validation examples. HateCheck is reserved for evaluation. The code uses the original three-class annotator majority before mapping `hatespeech → 1`, `normal/offensive → 0`.

The code also checks IDs, removes conflicting/duplicate training examples, filters cross-split normalized-text overlaps, and saves exact source revisions and hashes. The displayed counts are data-preparation results, **not model accuracy**. The full raw text and model weights will not be included in the small results ZIP.

In [ ]:
frames, hatecheck, audit = project.prepare_data(PROJECT_DIR)
print("Step 4 complete: datasets prepared. Actual filtered counts are saved.")

### Step 5 — Run the first before-and-after fine-tuning experiment
This is actual machine-learning training: **all model parameters are trainable**, not only the final classification layer.

The preset is DeHateBERT, two epochs, learning rate 0.00002, batch size 8, two-step gradient accumulation, and maximum length 128. The best of the two trained epochs is selected using HateXplain validation macro-F1. The released and selected fine-tuned model are then compared using the fixed 0.5 cutoff on HateCheck. Threshold/category outputs are also saved. Training history is saved after each epoch.

No improvement is guaranteed. Both lower false positives and potentially higher missed-hate rates must be reported. If this cell fails, do not assume it completed; go to Step 8 to save any partial outputs and the recorded error.

**Known limitation:** released models may have already seen some source data. Cardiff's training paper specifically includes HateXplain. Do not call validation scores independent unseen-data performance.

In [ ]:
first_result = project.run_model(PROJECT_DIR, frames, hatecheck,
                                 model_key="dehatebert")
print("First model complete. Continue to Step 6 to save the results.")

### Step 6 — Save the first model's results now
The results ZIP contains actual measurements and configuration files, not the raw offensive text or large checkpoints. Your browser should download a file beginning `Jaida_Update1_Results_`. Keep it. A trained checkpoint is still only in Colab's temporary `checkpoints/` folder unless you separately back it up.

In [ ]:
from google.colab import files
archive_path = project.export_results(PROJECT_DIR)
files.download(str(archive_path))

### Step 7 — Second model (optional for this first work session)
The final comparison plans both models. Getting the first model running provides initial real results for Update 1; the second can honestly remain upcoming.

This cell is OFF by default. To run the second model now, change only `RUN_SECOND_MODEL = False` to `True`, then run the cell. It uses the same data partitions and settings. Do not change settings based on which model performed better on HateCheck.

In [ ]:
RUN_SECOND_MODEL = False
if RUN_SECOND_MODEL:
    second_result = project.run_model(PROJECT_DIR, frames, hatecheck,
                                      model_key="roberta")
else:
    print("Second-model run skipped. Record it as upcoming, not completed.")

### Step 8 — Final export (also usable after an interrupted training cell)
Run this cell even when only the first model or data preparation finished. It exports the files that actually exist and does not invent missing results.

After downloading the ZIP, use **File → Download → Download .ipynb** to save this notebook with its outputs. Send the results ZIP back with the real GitHub repository link so the working paper can be updated with the measured results. **Do not submit the paper draft with pending results or a missing repository link.**

In [ ]:
from google.colab import files
archive_path = project.export_results(PROJECT_DIR)
files.download(str(archive_path))
print("Save this notebook too: File > Download > Download .ipynb.")

## What the files mean

`model_comparison.csv` — overall measured HateCheck scores. `fpr` and `fnr` are fractions; multiply by 100 for percentages.

`*_predictions.csv` — case IDs, true labels, model scores and predictions. No raw test sentences are exported.

`*_categories.csv` — performance by HateCheck functionality. A blank FPR/FNR means the denominator is absent, not zero errors.

`*_thresholds.csv` — fixed cutoff sensitivity from 0.1 to 0.9. Do not use these test results to claim an optimal deployment threshold.

`dehatebert/training_history.csv` and `run_config.json` — training evidence, selected epoch, settings, versions, revisions and GPU information. `status.json` distinguishes completed, running and failed work.

`data_audit.json` and `dataset_counts.csv` — actual post-filtering dataset counts and overlap checks.

`results_for_report.md` — a factual summary of only the outputs present, including before/after changes when both exist. It is not the finished ACL paper.

## Sources

- HateCheck paper: https://aclanthology.org/2021.acl-long.4/
- HateCheck data: https://github.com/paul-rottger/hatecheck-data
- HateXplain paper: https://doi.org/10.1609/aaai.v35i17.17745
- HateXplain data and original splits: https://github.com/hate-alert/HateXplain
- DeHateBERT: https://huggingface.co/Hate-speech-CNERG/dehatebert-mono-english
- Cardiff model: https://huggingface.co/cardiffnlp/twitter-roberta-base-hate-latest
- Cardiff training paper: https://aclanthology.org/2023.woah-1.25/

These sources describe the resources. They do not supply this project's experimental results.